# Course 5 — Generalized Linear Models: Poisson Regression

GLMs unify linear regression, logistic regression, and Poisson regression under one framework.
This course covers Poisson regression for count data — the log link that keeps predictions positive.

**Sections**
1. The GLM framework — distribution, link, linear predictor (0:00–0:20)
2. Why linear regression fails for counts (0:20–0:35)
3. Poisson regression — model, interpretation, MLE (0:35–1:00)
4. Bikeshare dataset demo with statsmodels (1:00–1:20)
5. Overdispersion — diagnosis and fixes (1:20–1:45)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

try:
    import statsmodels.api as sm
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print('statsmodels not installed — install with: pip install statsmodels')

print('Setup complete.')

## 1. The GLM Framework

A Generalized Linear Model has three components:

1. **Random component**: Y|X comes from an exponential family distribution
2. **Systematic component**: linear predictor η = β₀ + β₁x₁ + … + βₚxₚ  
3. **Link function**: g such that g(E[Y|X]) = η, equivalently E[Y|X] = g⁻¹(η)

| Distribution | Link g | E[Y\|X] | Method |
|---|---|---|---|
| Gaussian | Identity: μ | Xβ | Linear regression |
| Binomial | Logit: log(μ/(1−μ)) | σ(Xβ) | Logistic regression |
| Poisson | Log: log(μ) | exp(Xβ) | Poisson regression |
| Negative Binomial | Log: log(μ) | exp(Xβ) | Negative Binomial regression |

### Mean-variance relationships

| Distribution | Var(Y) | Implication |
|---|---|---|
| Gaussian | σ² (constant) | Equal spread everywhere (homoscedastic) |
| Poisson | μ | Variance grows with mean |
| Negative Binomial | μ + μ²/r | More spread than Poisson (overdispersed) |

All GLMs are fitted by **IRLS** (Iteratively Reweighted Least Squares) — a unified algorithm.
For Gaussian, IRLS converges in one step (it reduces to OLS).

In [ ]:
# Visualize the three link functions
mu_range = np.linspace(0.01, 10, 300)
eta_range = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Identity link (Gaussian / linear regression)
axes[0].plot(eta_range, eta_range, linewidth=2, color='C0')
axes[0].set_xlabel('η = Xβ'); axes[0].set_ylabel('E[Y|X]')
axes[0].set_title('Identity link (Gaussian)\nE[Y] = Xβ')
axes[0].set_ylim(-4, 4)

# Logit link (Binomial / logistic regression)
axes[1].plot(eta_range, 1/(1+np.exp(-eta_range)), linewidth=2, color='C1')
axes[1].set_xlabel('η = Xβ'); axes[1].set_ylabel('E[Y|X] = P(Y=1)')
axes[1].set_title('Logit link (Binomial)\nE[Y] = σ(Xβ) ∈ (0,1)')
axes[1].axhline(0.5, color='k', linestyle='--', alpha=0.4)

# Log link (Poisson)
axes[2].plot(eta_range, np.exp(eta_range), linewidth=2, color='C2')
axes[2].set_xlabel('η = Xβ'); axes[2].set_ylabel('E[Y|X] = count')
axes[2].set_title('Log link (Poisson)\nE[Y] = exp(Xβ) > 0 always')
axes[2].set_ylim(0, 30)

plt.tight_layout(); plt.show()

## 2. Why Linear Regression Fails for Count Data

In [ ]:
# Simulate count data: hospital arrivals per hour
rng = np.random.default_rng(42)
hour = np.arange(0, 24)
# True Poisson rate: low at night, peaks at noon and 6pm
true_rate = 5 + 15 * np.exp(-((hour - 12)**2)/18) + 10 * np.exp(-((hour - 18)**2)/10)
# Simulate 30 days
counts_all = rng.poisson(np.tile(true_rate, 30))
hours_all  = np.tile(hour, 30)

# Fit OLS
X_h = hours_all.reshape(-1, 1)
ols = LinearRegression().fit(X_h, counts_all)
xs_plot = np.arange(0, 24, 0.1).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(hours_all, counts_all, s=5, alpha=0.3, label='Observed counts')
axes[0].plot(xs_plot, ols.predict(xs_plot), color='C3', linewidth=2, label='OLS fit')
axes[0].plot(hour, true_rate, color='C2', linewidth=2, linestyle='--', label='True mean')
axes[0].set_xlabel('Hour of day'); axes[0].set_ylabel('Count')
axes[0].set_title('OLS on count data — misses the pattern')
axes[0].legend(fontsize=9)

# OLS residuals vs fitted — heteroscedastic
fitted_ols = ols.predict(X_h)
resid_ols = counts_all - fitted_ols
axes[1].scatter(fitted_ols, resid_ols, s=5, alpha=0.3)
axes[1].axhline(0, color='k', linestyle='--')
axes[1].set_xlabel('Fitted values'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs fitted — funnel shape = heteroscedasticity')
plt.tight_layout(); plt.show()

neg_count = (ols.predict(xs_plot) < 0).sum()
print(f'OLS predicts negative counts at {neg_count} out of {len(xs_plot)} grid points')

## 3. Poisson Regression

### The model

$$\log(E[Y | X]) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

$$E[Y | X] = \exp(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p) > 0 \text{ always}$$

With $Y | X \sim \text{Poisson}(\exp(X\beta))$.

### Coefficient interpretation

- $e^{\beta_j}$ = multiplicative change in mean count per unit increase in $x_j$
- $\beta_j = 0$ → no effect ($e^0 = 1$)
- $\beta_j = 0.10$ → $e^{0.10} \approx 1.11$ → +11% increase per unit
- $\beta_j = \log(2) \approx 0.69$ → count doubles per unit

### Log-likelihood

$$\ell(\beta) = \sum_i \left[ y_i (X_i \beta) - \exp(X_i \beta) - \log(y_i!) \right]$$

This is **concave** in β — any optimizer finds the unique global optimum.
The $\log(y_i!)$ term is a constant w.r.t. β — it drops out of optimization.

In [ ]:
# Fit Poisson regression (sklearn) vs OLS on simulated count data
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import Pipeline

# Use spline features to capture non-linear relationship with hour
pipe_ols  = Pipeline([('spline', SplineTransformer(n_knots=8, degree=3)),
                       ('ols', LinearRegression())])
pipe_pois = Pipeline([('spline', SplineTransformer(n_knots=8, degree=3)),
                       ('pois', PoissonRegressor(alpha=0.01, max_iter=1000))])

pipe_ols.fit(X_h, counts_all)
pipe_pois.fit(X_h, counts_all)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(hours_all, counts_all, s=5, alpha=0.2, label='Observed counts')
ax.plot(xs_plot, pipe_ols.predict(xs_plot), color='C3', linewidth=2, label='OLS + splines')
ax.plot(xs_plot, pipe_pois.predict(xs_plot), color='C2', linewidth=2, label='Poisson + splines')
ax.plot(hour, true_rate, color='gray', linewidth=2, linestyle=':', label='True mean')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Count')
ax.set_title('Poisson regression vs OLS — Poisson stays non-negative')
ax.legend(); plt.show()

# Check: any negative predictions from Poisson?
pois_preds = pipe_pois.predict(xs_plot)
print(f'Poisson negative predictions: {(pois_preds < 0).sum()} (should be 0)')
print(f'OLS negative predictions:     {(pipe_ols.predict(xs_plot) < 0).sum()}')

In [ ]:
# Simple Poisson regression: demonstrate coefficient interpretation
from sklearn.linear_model import PoissonRegressor

# Simulated: insurance claims ~ age + smoker
rng2 = np.random.default_rng(7)
n = 500
age = rng2.uniform(18, 70, n)
smoker = rng2.binomial(1, 0.3, n).astype(float)
log_rate = 0.5 + 0.02 * age + 0.8 * smoker  # true: age adds ~2%, smoker doubles
claims = rng2.poisson(np.exp(log_rate))

X_ins = np.column_stack([age, smoker])
pois_ins = PoissonRegressor(alpha=0, max_iter=2000).fit(X_ins, claims)
print('Poisson regression: insurance claims ~ age + smoker')
print(f'Intercept β₀ = {pois_ins.intercept_:.4f}')
print(f'β_age        = {pois_ins.coef_[0]:.4f}  → e^β = {np.exp(pois_ins.coef_[0]):.4f}  (+{(np.exp(pois_ins.coef_[0])-1)*100:.1f}% per year)')
print(f'β_smoker     = {pois_ins.coef_[1]:.4f}  → e^β = {np.exp(pois_ins.coef_[1]):.4f}  (smokers have {np.exp(pois_ins.coef_[1]):.2f}× more claims)')
print(f'\nTrue values: β_age≈0.02 (e^β≈1.02), β_smoker≈0.80 (e^β≈2.23)')

## 4. Bikeshare Dataset Demo

In [ ]:
# Load Bikeshare dataset (from ISLR)
# Try to load from shared data_utils; fall back to seaborn or sklearn
try:
    from shared.data_utils import load_dataset
    bikes = load_dataset('bikeshare')
    print('Loaded from shared data_utils')
except Exception:
    # Generate a realistic simulation of the Bikeshare dataset
    rng3 = np.random.default_rng(0)
    n_hours = 8760  # one year
    hr = np.tile(np.arange(24), n_hours // 24)
    mnth = np.repeat(np.arange(1, 13), n_hours // 12)
    workingday = rng3.binomial(1, 0.7, n_hours).astype(float)
    temp = 0.5 + 0.4 * np.sin(2 * np.pi * mnth / 12) + 0.05 * rng3.standard_normal(n_hours)
    temp = np.clip(temp, 0, 1)
    weathersit = rng3.choice([1,2,3,4], n_hours, p=[0.63, 0.24, 0.10, 0.03])
    # True Poisson rate
    log_mu = (3.0 
              + 0.8 * temp
              - 0.3 * (weathersit >= 3).astype(float)
              + 0.5 * np.where(hr == 8, 1, 0)
              + 0.8 * np.where(hr == 17, 1, 0)
              + 0.4 * np.where((hr >= 9) & (hr <= 16), 1, 0)
              - 1.5 * np.where(hr <= 4, 1, 0))
    bikers = rng3.negative_binomial(5, 5/(5+np.exp(log_mu)))  # overdispersed
    bikes = pd.DataFrame({
        'hr': hr, 'mnth': mnth, 'workingday': workingday,
        'temp': temp, 'weathersit': weathersit, 'bikers': bikers
    })
    print('Using simulated Bikeshare data (realistic)')

print(bikes.shape)
print(bikes[['bikers']].describe().round(2))

In [ ]:
# Exploratory: bike counts by hour and month
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
by_hour = bikes.groupby('hr')['bikers'].mean()
by_month = bikes.groupby('mnth')['bikers'].mean()

axes[0].bar(by_hour.index, by_hour.values, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Hour of day'); axes[0].set_ylabel('Mean rentals')
axes[0].set_title('Average bike rentals by hour')
axes[0].set_xticks(range(0, 24, 2))

axes[1].bar(by_month.index, by_month.values, color='C2', alpha=0.8)
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Mean rentals')
axes[1].set_title('Average bike rentals by month')
plt.tight_layout(); plt.show()

# Check mean-variance relationship (key diagnostic)
by_hr_stats = bikes.groupby('hr')['bikers'].agg(['mean', 'var'])
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(by_hr_stats['mean'], by_hr_stats['var'], s=60)
ax.plot([0, by_hr_stats['mean'].max()], [0, by_hr_stats['mean'].max()],
        'k--', alpha=0.5, label='Var = Mean (Poisson)')
ax.set_xlabel('Mean by hour'); ax.set_ylabel('Variance by hour')
ax.set_title('Mean-Variance plot: variance >> mean → overdispersed')
ax.legend(); plt.show()

In [ ]:
# Fit Poisson regression with statsmodels (for coefficient table)
bikes_model = bikes.copy()
# One-hot encode hour and month (drop first for identification)
bikes_model = pd.get_dummies(bikes_model, columns=['hr', 'mnth', 'weathersit'],
                              drop_first=True, dtype=float)
X_cols = [c for c in bikes_model.columns if c != 'bikers']
X_bk = bikes_model[X_cols]
y_bk = bikes_model['bikers']

if HAS_SM:
    X_const = sm.add_constant(X_bk)
    pois_fit = sm.GLM(y_bk, X_const, family=sm.families.Poisson()).fit()
    # Show key coefficients only
    summary_df = pois_fit.summary2().tables[1]
    key_rows = [r for r in summary_df.index if any(s in r for s in ['temp','weathersit','workingday','const'])]
    print('Poisson GLM — key coefficients:')
    print(summary_df.loc[key_rows, ['Coef.','Std.Err.','z','P>|z|']].round(4))
    print(f'\nNull deviance:     {pois_fit.null_deviance:.1f}')
    print(f'Residual deviance: {pois_fit.deviance:.1f}')
    print(f'Pearson χ²/df:     {pois_fit.pearson_chi2 / pois_fit.df_resid:.2f}  (>1 → overdispersed)')
else:
    from sklearn.linear_model import PoissonRegressor
    from sklearn.preprocessing import StandardScaler
    pois_sk = PoissonRegressor(alpha=0, max_iter=3000)
    pois_sk.fit(X_bk, y_bk)
    print('sklearn PoissonRegressor (no p-values; use statsmodels for inference)')
    key_feats = ['temp', 'workingday']
    for f in key_feats:
        if f in X_bk.columns:
            idx = list(X_bk.columns).index(f)
            beta = pois_sk.coef_[idx]
            print(f'  {f}: β={beta:.4f}, e^β={np.exp(beta):.4f}')

In [ ]:
# Compare Poisson vs OLS predictions by hour
from sklearn.linear_model import LinearRegression, PoissonRegressor

ols_sk   = LinearRegression().fit(X_bk, y_bk)
pois_sk2 = PoissonRegressor(alpha=0, max_iter=3000).fit(X_bk, y_bk)

bikes_model['pred_ols']    = ols_sk.predict(X_bk)
bikes_model['pred_poisson'] = pois_sk2.predict(X_bk)
bikes_model['hr_orig'] = bikes['hr'].values

by_hr_compare = bikes_model.groupby('hr_orig')[['bikers','pred_ols','pred_poisson']].mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(by_hr_compare.index, by_hr_compare['bikers'], 'ko-', markersize=5, label='Observed mean')
ax.plot(by_hr_compare.index, by_hr_compare['pred_ols'], 'C3--', linewidth=2, label='OLS prediction')
ax.plot(by_hr_compare.index, by_hr_compare['pred_poisson'], 'C2-', linewidth=2, label='Poisson prediction')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Mean bike rentals')
ax.set_title('Poisson vs OLS predictions by hour (averaged over all days)')
ax.set_xticks(range(0, 24, 2)); ax.legend(); plt.show()

# RMSE comparison
rmse_ols  = np.sqrt(mean_squared_error(y_bk, bikes_model['pred_ols']))
rmse_pois = np.sqrt(mean_squared_error(y_bk, bikes_model['pred_poisson']))
print(f'RMSE OLS:     {rmse_ols:.2f}')
print(f'RMSE Poisson: {rmse_pois:.2f}')
print(f'\nNegative OLS predictions: {(bikes_model["pred_ols"] < 0).sum()}')
print(f'Negative Poisson preds:   {(bikes_model["pred_poisson"] < 0).sum()}')

## 5. Overdispersion — Diagnosis and Fixes

The Poisson distribution assumes **Var(Y) = E[Y]**. Real count data often has much higher variance.

**Causes of overdispersion**:
- Unobserved heterogeneity (missing covariates)
- Clustering / correlation within groups
- Excess zeros (zero-inflation)
- True count process is negative binomial

**Diagnosis**: Pearson χ² statistic / residual df. If >> 1, the model is overdispersed.

**Consequence**: standard errors are too small → confidence intervals too narrow → false positives.

**Fixes**:
1. **Quasi-Poisson**: same β̂, but scale SEs by √(Pearson χ²/df)
2. **Negative Binomial**: explicitly models extra-Poisson dispersion with parameter r
3. **Zero-Inflated Poisson (ZIP)**: mixture of Poisson and point mass at zero

In [ ]:
# Diagnose overdispersion
preds_pois = pois_sk2.predict(X_bk)
pearson_resid = (y_bk - preds_pois) / np.sqrt(preds_pois)
pearson_chi2 = (pearson_resid**2).sum()
df_resid = len(y_bk) - X_bk.shape[1] - 1
dispersion = pearson_chi2 / df_resid

print(f'Pearson χ² = {pearson_chi2:.1f}')
print(f'df residual = {df_resid}')
print(f'Dispersion ratio = {dispersion:.2f}  (>>1 → overdispersed)')
print(f'\nQuasi-Poisson SE inflation factor: √{dispersion:.2f} = {np.sqrt(dispersion):.3f}')
print('All Poisson p-values should be widened by this factor.')

# Visualize Pearson residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(preds_pois, pearson_resid, s=3, alpha=0.3)
axes[0].axhline(0, color='k', linestyle='--')
axes[0].set_xlabel('Fitted count'); axes[0].set_ylabel('Pearson residual')
axes[0].set_title('Pearson residuals vs fitted — fan shape = overdispersion')
axes[1].hist(pearson_resid, bins=60, density=True, alpha=0.7)
from scipy.stats import norm as sp_norm
x_norm = np.linspace(-5, 5, 200)
axes[1].plot(x_norm, sp_norm.pdf(x_norm), 'C3-', linewidth=2, label='N(0,1)')
axes[1].set_xlabel('Pearson residual'); axes[1].set_ylabel('Density')
axes[1].set_title('Heavy tails → overdispersion')
axes[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Negative Binomial regression with statsmodels
if HAS_SM:
    try:
        nb_fit = sm.GLM(y_bk, sm.add_constant(X_bk),
                        family=sm.families.NegativeBinomial()).fit()
        print('Negative Binomial GLM:')
        print(f'  Deviance: {nb_fit.deviance:.1f}  (vs Poisson: {pois_fit.deviance:.1f})')
        print(f'  AIC:      {nb_fit.aic:.1f}  (vs Poisson: {pois_fit.aic:.1f})')
        print('  Lower AIC = better fit. Negative Binomial should win here.')
    except Exception as e:
        print(f'NB fit: {e}')
        print('For Negative Binomial, try: statsmodels.discrete.discrete_model.NegativeBinomial')
else:
    print('statsmodels not available for Negative Binomial. Install with: pip install statsmodels')

## Recap

1. **GLM = distribution + link + linear predictor.** Unifies linear, logistic, and Poisson regression.
2. **Log link guarantees positive predictions** — no negative counts ever.
3. **e^β_j = multiplicative effect on mean count** per unit increase of x_j.
4. **Poisson MLE is concave** — any optimizer finds the unique global optimum.
5. **Deviance** plays the role of RSS; AIC = deviance + 2p for model selection.
6. **Check overdispersion**: Pearson χ²/df >> 1 → use quasi-Poisson or Negative Binomial.

---

# Exercises

## Exercise 1

**Task 1.** Using the Bikeshare data, fit a Poisson regression on just `hr` (as numeric) and `temp`.
Report the coefficient table and interpret `β_temp` in terms of multiplicative effect.

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
X_simple = bikes[['hr','temp']].to_numpy()
y_simple = bikes['bikers'].to_numpy()
m_simple = PoissonRegressor(alpha=0, max_iter=2000).fit(X_simple, y_simple)
print(f'Intercept β₀ = {m_simple.intercept_:.4f}')
for name, beta in zip(['hr','temp'], m_simple.coef_):
    print(f'β_{name:5s} = {beta:.4f}  → e^β = {np.exp(beta):.4f}')
print(f'\nInterpretation: a one-unit increase in temp multiplies expected rentals by {np.exp(m_simple.coef_[1]):.2f}×')

## Exercise 2

**Task 2.** Compare train RMSE between OLS and PoissonRegressor on the full Bikeshare feature set.
Plot predicted vs observed for both.

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_bk, y_bk, test_size=0.3, random_state=0)
ols_b  = LinearRegression().fit(Xtr_b, ytr_b)
pois_b = PoissonRegressor(alpha=0, max_iter=3000).fit(Xtr_b, ytr_b)

for name, model in [('OLS', ols_b), ('Poisson', pois_b)]:
    preds = model.predict(Xte_b)
    rmse = np.sqrt(mean_squared_error(yte_b, preds))
    neg  = (preds < 0).sum()
    print(f'{name}: RMSE = {rmse:.2f}, negative predictions = {neg}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, model) in zip(axes, [('OLS', ols_b), ('Poisson', pois_b)]):
    preds = model.predict(Xte_b)
    ax.scatter(yte_b, preds, s=3, alpha=0.3)
    lim = max(yte_b.max(), preds.max())
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.5)
    ax.set_xlabel('Observed'); ax.set_ylabel('Predicted')
    ax.set_title(f'{name}: predicted vs observed')
plt.tight_layout(); plt.show()

## Exercise 3

**Task 3.** Compute the overdispersion ratio for your fitted Poisson model.
By how much should the standard errors be inflated?

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
preds_te = pois_b.predict(Xte_b)
preds_te_safe = np.maximum(preds_te, 1e-8)  # avoid log(0)
pearson_resid_te = (yte_b - preds_te_safe) / np.sqrt(preds_te_safe)
disp = (pearson_resid_te**2).sum() / (len(yte_b) - Xte_b.shape[1] - 1)
print(f'Overdispersion ratio: {disp:.2f}')
print(f'SE inflation factor:  {np.sqrt(disp):.3f}')
print('Standard Poisson SEs are underestimated by this factor.')